# IRI v2 Filesystem Async-Task Smoke Test

The same test as [filesystem.ipynb](filesystem.ipynb) (IRI API **v1**), written for the IRI Facility API **v2** (`/api/v2`).

It will:
- Discover projects/capabilities/resources
- Prompt you to select a **storage resource** for test
- Resolve your username (`whoami`) and a writable directory (`storage/locations`) on it
- Run a sandboxed sequence: `mkdir`, `upload`, `file`, `stat`, `ls`, `chmod`, `head`, `tail`, `view`, `checksum`, `cp`, `mv`, `symlink`, `compress`, `extract`, `download`, `rm`

## What is different in v2

| Operation | v1 | v2 (this notebook) |
|---|---|---|
| `file`, `stat`, `ls`, `head`, `tail`, `view`, `checksum`, `download` | `GET` + query parameters | `POST` + JSON body (same field names) |
| `chmod`, `chown` | `PUT` + JSON body | `POST` + JSON body |
| `rm` | `DELETE` + `?path=` | `POST` + JSON body |
| `mkdir`, `upload`, `cp`, `mv`, `symlink`, `compress`, `extract` | unchanged | unchanged method; source is `path` (`source_path` is only an undocumented alias) |
| `compress` / `extract` compression | `gzip` | URN: `urn:doe-iri:compression:gzip` |
| `resource_type` | `storage`, `compute`, ... | URN, e.g. `urn:doe-iri:resource:storage:system` |
| Which resources support filesystem operations | guess from the resource type | HAL `_links` on the resource, e.g. `iri:list-directory` (shown as "v2 ops") |
| Username / test directory | relative paths | `GET /account/whoami` + `GET /storage/locations/{resource_id}` |

Facilities serving IRI v2: ESnet East, ESnet West, NERSC. ALCF and NRP serve v1 only: use
[filesystem.ipynb](filesystem.ipynb) for them.

## Before you run
Use the [login-globus](login-globus.ipynb) or [login-esnet](login-esnet.ipynb) notebook to receive a token. The same
token works for v1 and v2.

## 1) Imports and load env

In [ ]:
!pip install python-dotenv

In [ ]:
import os
import sys
import time
import json
import datetime as dt
import requests
from pathlib import Path
from typing import List, Optional

from dotenv import load_dotenv

load_dotenv()
print("Environment loaded")

## 2) Configuration
Edit `BASE_URL` or override `IRI_API_TOKEN` in your environment.

In [ ]:
BASE_URL = "https://iri-dev.ppg.es.net/api/v2" # Esnet East
#BASE_URL = "https://esnet-west.sdn-sense.net/api/v2" # Esnet West
#BASE_URL = "https://api.iri.nersc.gov/api/v2" # NERSC
# ALCF and NRP serve IRI v1 only: use the v1 notebooks for them.

home_path = Path("~/.iri_token.json").expanduser()
TOKEN = os.environ.get("IRI_API_TOKEN")
if TOKEN:
    print(f"IMPORTANT: Token defined inside .env file (or as environment variable). WILL NOT LOAD TOKEN from {home_path}")
else:
    # Load token from home_path
    with open(home_path) as f:
        tokendict = json.load(f)
        TOKEN = tokendict.get("IRI_API_TOKEN")
# If the token is still not available. print warning!
if not TOKEN:
    print(f"WARNING! Was unable to receive token from env variable or {home_path}. Use login-globus or login-esnet notebook to receive a token.")
    print("WARNING! FURTHER COMMANDS WILL NOT SUCCEED.")
# Unless set here (or in .env), the username comes from GET /account/whoami and the test
# directory is created under the writable location from GET /storage/locations (both v2).
USERNAME = os.environ.get("IRI_USERNAME")
TEST_BASE_DIR = os.environ.get("IRI_FS_TEST_DIR")

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/json"}
POLL_INTERVAL = 2
TIMEOUT = 180

## 3) Helper functions
Table formatting + small utilities.

In [ ]:
def die(msg: str) -> None:
    """Print error message and exit."""
    print(f"\nERROR: {msg}")
    raise SystemExit(1)


def print_table(headers, rows):
    """Pretty-print a table with dynamic column width."""
    widths = [len(h) for h in headers]

    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))

    fmt = " | ".join(f"{{:<{w}}}" for w in widths)
    sep = "-+-".join("-" * w for w in widths)

    print(fmt.format(*headers))
    print(sep)
    for row in rows:
        print(fmt.format(*[str(c) for c in row]))
    print()


def cap_name(allcaps, uri):
    """Get capability name from URI, or return URI if not found."""
    c = allcaps.get(uri)
    return c.get("name") if c else uri


# v2: resources advertise the operations they support as HAL _links (https://iri.science/rels/<rel>).
V2_RELS = {"compute": "iri:submit-job", "filesystem": "iri:list-directory"}


def v2_ops(resource: dict) -> str:
    """Which of the smoke test's operations a resource advertises in its v2 _links."""
    links = resource.get("_links") or {}
    return ", ".join(kind for kind, rel in V2_RELS.items() if rel in links)

## 4) Discovery: projects, capabilities, allocations, resources
Lists what you can access and returns matching storage resource IDs.

In [ ]:
def getAllResources():
    """Get and print all storage resources available to the user, along with their capabilities."""
    print("\n" + "="*40)
    print("=== DISCOVERING RESOURCES AND CAPABILITIES ===")

    # -----------------------------
    # Projects
    # -----------------------------
    projects = requests.get(f"{BASE_URL}/account/projects", headers=HEADERS, timeout=TIMEOUT).json()
    project_rows = [[p.get("id"), p.get("name", ""), p.get("description", "")] for p in projects]

    print("\n=== PROJECTS ===")
    print_table(["Project ID", "Name", "Description"], project_rows)

    # -----------------------------
    # Capabilities
    # -----------------------------
    caps = requests.get(f"{BASE_URL}/account/capabilities", headers=HEADERS, timeout=TIMEOUT).json()
    cap_rows = [[c.get("self_uri"), c.get("name"), c.get("description", "")] for c in caps]
    cap_by_uri = {c["self_uri"]: c for c in caps}

    print("\n=== CAPABILITIES ===")
    print_table(["Capability URI", "Name", "Description"], cap_rows)

    # -----------------------------
    # Allocations per project
    # -----------------------------
    alloc_rows = []
    projectStorageCaps = set()

    for pr in projects:
        allocs = requests.get(
            f"{BASE_URL}/account/projects/{pr['id']}/project_allocations",
            headers=HEADERS,
            timeout=TIMEOUT,
        ).json()

        for a in allocs:
            alloc_rows.append([pr["id"], a.get("id"), cap_name(cap_by_uri, a.get("capability_uri"))])

            if a.get("capability_uri") in cap_by_uri:
                projectStorageCaps.add(a["capability_uri"])

    print("\n=== PROJECT ALLOCATIONS ===")
    print_table(["Project ID", "Allocation ID", "Capability URI"], alloc_rows)

    if not projectStorageCaps:
        die("No storage allocations found")

    # -----------------------------
    # Resources
    # -----------------------------
    resources = requests.get(
        f"{BASE_URL}/status/resources?offset=0&limit=100",
        headers=HEADERS,
        timeout=TIMEOUT,
    ).json()

    resource_rows = []
    matching = []

    for r in resources:
        caps = r.get("capability_uris", [])
        cap_names = [cap_name(cap_by_uri, c) for c in caps]

        match = any(cap in caps for cap in projectStorageCaps)

        if match:
            resource_rows.append(
                [
                    r.get("id"),
                    r.get("name", ""),
                    r.get("resource_type", ""),
                    r.get("description", ""),
                    ", ".join(cap_names),
                    r.get("current_status", ""),
                    v2_ops(r),
                ]
            )
            matching.append(r["id"])

    print("\n=== RESOURCES (v2 ops: operations advertised in the resource _links; pick one with 'filesystem') ===")
    print_table(["Resource ID", "Name", "Type", "Description", "Capability URIs", "Current Status", "v2 ops"], resource_rows)

    if not matching:
        die("No storage resources found")

    return matching

## 5) Async task API helpers
Submit a task and poll it to completion.

In v2 every filesystem operation is a `POST` with a JSON body; `fs_task` wraps submit + wait.

In [ ]:
def submit(method, path, **kwargs):
    """Submit a task and return the task envelope (task_id, task_uri)."""
    print(f"Submitting {method} {path} with {kwargs}")
    url = f"{BASE_URL}{path}"
    r = requests.request(method, url, headers=HEADERS, timeout=TIMEOUT, **kwargs)

    if not r.ok:
        die(f"{method} {url} failed: {r.status_code} {r.text}")

    data = r.json()
    if not data.get("task_id"):
        die(f"No task_id in response: {data}")
    if not data.get("task_uri"):
        die(f"No task_uri in response: {data}")

    return data


def wait_task(taskin):
    """Wait for a task to complete and return its result."""
    deadline = time.time() + TIMEOUT

    while time.time() < deadline:
        r = requests.get(taskin["task_uri"], headers=HEADERS, timeout=TIMEOUT)

        if not r.ok:
            die(f"Task query failed: {r.status_code} {r.text}")

        t = r.json()
        status = t["status"]

        print(f"   Task {taskin['task_id']}: {status}")

        if status == "completed":
            print(f"   Task result: {t.get('result')}")
            return t.get("result")

        if status in ("failed", "canceled"):
            die(f"Task {taskin['task_id']} ended with status {status}: {t}")

        time.sleep(POLL_INTERVAL)

    die(f"Task {taskin['task_id']} timed out")


def fs_task(op, resource_id, **body):
    """Run a v2 filesystem operation (POST /filesystem/<op>/<resource_id> with a JSON body) and wait for its result."""
    return wait_task(submit("POST", f"/filesystem/{op}/{resource_id}", json=body))


def resolve_writable_dir(resource_ids: List[str]) -> Optional[str]:
    """Writable directory for this user from GET /storage/locations (v2): home first, else any writable location."""
    for rid in resource_ids:
        r = requests.get(f"{BASE_URL}/storage/locations/{rid}", headers=HEADERS, params={"intent": "write"}, timeout=TIMEOUT)
        if not r.ok:
            print(f"   storage/locations/{rid}: {r.status_code} {r.text[:200]}")
            continue
        locations = r.json()
        print(f"\n=== STORAGE LOCATIONS ({rid}) ===")
        print_table(
            ["Logical name", "Path", "Filesystem", "Performance", "Shared", "Read", "Write"],
            [
                [loc.get("logical_name"), loc.get("path"), loc.get("filesystem") or "", loc.get("performance_tier") or "",
                 loc.get("shared"), (loc.get("access") or {}).get("read"), (loc.get("access") or {}).get("write")]
                for loc in locations
            ],
        )
        writable = [loc for loc in locations if (loc.get("access") or {}).get("write")]
        for loc in sorted(writable, key=lambda loc: loc.get("logical_name") != "home"):
            return loc["path"]
    return None

## 6) Who am I (v2)
`GET /account/whoami` (new in v2) returns your username at the facility. It is used for the default job directory
below, instead of a hardcoded username. Set `IRI_USERNAME` to override it.

In [ ]:
r = requests.get(f"{BASE_URL}/account/whoami", headers=HEADERS, timeout=TIMEOUT)
if not r.ok:
    die(f"whoami failed: {r.status_code} {r.text}")
whoami = r.json()
print("Facility username (whoami):", whoami.get("username"))
if not USERNAME:
    USERNAME = whoami["username"]
print("USERNAME:", USERNAME)

## 7) Run the filesystem smoke test
Execute the full end-to-end sequence.

In [ ]:
matching_ids = getAllResources()

print("\n" + "="*40)
RESOURCE_ID = input("Enter the ID of the storage resource to test against:\n").strip()
print("Chosen storage resource ID:", RESOURCE_ID)

timestamp = dt.datetime.now(dt.UTC).strftime("%Y%m%d-%H%M%S")
# v2: create the test directory under a writable location from storage/locations
# (v1 used a path relative to the facility's default directory).
if not TEST_BASE_DIR:
    TEST_BASE_DIR = resolve_writable_dir([RESOURCE_ID])
    if not TEST_BASE_DIR:
        print("No writable location from storage/locations; using a path relative to the facility's default directory (as in v1).")
base_dir = f"{TEST_BASE_DIR.rstrip('/')}/iri-fs-test-{timestamp}" if TEST_BASE_DIR else f"iri-fs-test-{timestamp}"
print("Test directory:", base_dir)
file_path = f"{base_dir}/hello.txt"
copy_path = f"{base_dir}/hello_copy.txt"
moved_path = f"{base_dir}/hello_moved.txt"
link_path = f"{base_dir}/hello_link.txt"
archive_path = f"{base_dir}.tar.gz"
extract_dir = f"{base_dir}_extracted"
content = f"hello world {timestamp}\n"

## 8) Create directory

In [ ]:
print("\n" + "="*40)
print("=== CREATE DIRECTORY ===")
fs_task("mkdir", RESOURCE_ID, path=base_dir, parent=True)

## 9) Upload file

In [ ]:
print("\n" + "="*40)
print("=== UPLOAD FILE ===")
task = submit("POST", f"/filesystem/upload/{RESOURCE_ID}?path={file_path}", files={"file": ("hello.txt", content.encode())})
wait_task(task)

## 10) Get File type

In [ ]:
print("\n" + "="*40)
print("=== FILE TYPE ===")
fs_task("file", RESOURCE_ID, path=file_path)

## 11) Get File stats

In [ ]:
print("\n" + "="*40)
print("=== STAT ===")
fs_task("stat", RESOURCE_ID, path=file_path)

## 12) List files

In [ ]:
print("\n" + "="*40)
print("=== LS ===")
fs_task("ls", RESOURCE_ID, path=base_dir)

## 13) Chmod file

In [ ]:
print("\n" + "="*40)
print("=== CHMOD ===")
fs_task("chmod", RESOURCE_ID, path=file_path, mode="644")

## 14) Execute file head

In [ ]:
print("\n" + "="*40)
print("=== HEAD ===")
fs_task("head", RESOURCE_ID, path=file_path, lines=1)

## 15) Execute file tail

In [ ]:
print("\n" + "="*40)
print("=== TAIL ===")
fs_task("tail", RESOURCE_ID, path=file_path, lines=1)

## 16) Execute file view

In [ ]:
print("\n" + "="*40)
print("=== VIEW ===")
fs_task("view", RESOURCE_ID, path=file_path, size=4096, offset=0)

## 17) Get file checksum

In [ ]:
print("\n" + "="*40)
print("=== CHECKSUM ===")
fs_task("checksum", RESOURCE_ID, path=file_path)

## 18) Execute file copy

In [ ]:
print("\n" + "="*40)
print("=== COPY FILE ===")
fs_task("cp", RESOURCE_ID, path=file_path, target_path=copy_path)

## 19) Execute file move

In [ ]:
print("\n" + "="*40)
print("=== MOVE FILE ===")
fs_task("mv", RESOURCE_ID, path=copy_path, target_path=moved_path)

## 20) Create symlink

In [ ]:
print("\n" + "="*40)
print("=== CREATE SYMLINK ===")
fs_task("symlink", RESOURCE_ID, path=moved_path, link_path=link_path)

## 21) Compress directory

In [ ]:
print("\n" + "="*40)
print("=== COMPRESS DIRECTORY ===")
fs_task("compress", RESOURCE_ID, path=base_dir, target_path=archive_path, compression="urn:doe-iri:compression:gzip")

## 22) Extract compressed file

In [ ]:
print("\n" + "="*40)
print("=== EXTRACT ARCHIVE ===")
fs_task("extract", RESOURCE_ID, path=archive_path, target_path=extract_dir, compression="urn:doe-iri:compression:gzip")

## 23) Download file to local environment

In [ ]:
print("\n" + "="*40)
print("=== DOWNLOAD FILE ===")
fs_task("download", RESOURCE_ID, path=moved_path)

## 24) Cleanup (remove files)

In [ ]:
print("\n" + "="*40)
print("=== CLEANUP ===")
for p in [base_dir, archive_path, extract_dir]:
    fs_task("rm", RESOURCE_ID, path=p)